## Data Ingestion to Bronze Layer

- Address
- Customer
- Person
- Product
- ProductCategory
- ProductDescription
- ProductModel
- SalesOrderdetail
- SalesOrderHeader
- Store


In [0]:
from pyspark.sql import SparkSession, DataFrame
import requests
import json

def retrieve_api_data(endpoint: str)-> json:

    # Define the API endpoint, and headers
    base_url = "https://demodata.grapecity.com/adventureworks/api/v1/"
    api_url = f"{base_url}{endpoint}"
    headers = {
        "Accept": "application/json"
    }

    # Check if the request was successful
    try:
        # Make a GET request to the API
        response = requests.get(api_url, headers = headers)
        response.raise_for_status()
        # Parse the JSON response
        payload = response.json()

        print(f"Successfully retrieve json_data from {api_url}")
    # Exception Handling for API Request
    except requests.exceptions.RequestException as e:
        print(f"Failed to retrieve json_data from {api_url} - {e}")

    return payload




convert_json_to_dataframe allows to convert the json_data into dataframe using PySpark


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, schema_of_json, from_json, lit
import pyspark.sql.functions as sf
import json, traceback


def convert_json_to_dataframe():

    spark = SparkSession.builder.appName("Ingest to Bronze Layer")\
        .getOrCreate()

    # Returns list of tables based on endpoint
    tbl_list = [
        "addresses", "customers", "products", "productCategories", "productDescriptions",
        "productModels", "salesOrderDetails", "salesOrders", "stores", "persons"
    ]

    # api_schemas_list = retrieve_schemas()
    for tbl in tbl_list:
        print(f"Processing {tbl} endpoint...")
        api_data = retrieve_api_data(tbl)
        # Create df_spark dataframe for Spark
        print(f"Converting {tbl} json data to DataFrame...")

        try:

            # df_spark = spark.createDataFrame(api_data)
            # df_spark = spark.createDataFrame([json.dumps(api_data)])
            
        # Convert JSON data to Spark DataFrame
            # if isinstance(api_data, list):
            #     df_spark = spark.createDataFrame(api_data)
            # elif isinstance(api_data, dict) and 'value' in api_data:
            #     df_spark = spark.createDataFrame(api_data['value'])
            # else:
                # Convert JSON objects to strings
            json_strings = [json.dumps(record) for record in api_data]
            df_spark = spark.createDataFrame(json_strings, "string").toDF("raw_json")
            # Infer schema from a sample
            sample_json_data = df_spark.select("raw_json").head()[0]
            inferred_schema = spark.range(1).select(schema_of_json(lit(sample_json_data))).collect()[0][0]
        
            # Parse JSON strings into structured DataFrame
            df_parsed = df_spark.withColumn("parsed", from_json("raw_json", inferred_schema))
            df_structured = df_parsed.select("parsed.*")

            # return df_spark

            
            df_structured.printSchema()
            print(f"{tbl} json data has been converted to DataFrame...")
            # df_spark.show(truncate=False)
            write_to_bronze_layer(df_structured, tbl)
            

        except AssertionError as e:
            print(f"Error converting {tbl} json data to DataFrame - ", str(e.args))
            traceback.print_exc(e)
            



In [0]:
from datetime import datetime
import pytz

def get_current_timestamp()-> datetime:
    # Get the current time in UTC
    current_time_utc = datetime.now(pytz.utc)
    # Define the timezone for Western Australia
    wa_tz = pytz.timezone('Australia/Perth')

    # Get the current time in UTC and convert it to AWST
    current_time_wa = datetime.now(wa_tz)

    # Print the current timestamp
    print("Current timestamp in AWST:", current_time_wa.strftime('%Y-%m-%d %H:%M:%S'))

    return current_time_wa

In [0]:
from pyspark.sql import DataFrame
from datetime import datetime
import traceback
import pytz

def write_to_bronze_layer(df: DataFrame, tbl_name: str):
    if df.count == 0:
        print(f"Empty DataFrame")
    else:
        # print(df)

        schema = "bronze_adworks_jaq_test"  # Bronze schema name
        catalog = "adventureworks_dev"  # Unity Catalog name
        tbl_full_name = f"{catalog}.{schema}.{tbl_name}"

        try:

            if(tbl_name == "persons"):
                    df = df.select(
                        col("personId"),
                        col("personType"),
                        col("nameStyle"),
                        col("title"),
                        col("firstName"),
                        col("middleName"),
                        col("lastName"),
                        col("suffix"),
                        col("emailPromotion"),
                        col("additionalContactInfo"),
                        col("demographics.TotalPurchaseYTD").alias("TotalPurchaseYTD"),
                        col("modifiedDate")
                    )
            elif (tbl_name == "stores"):
                df = df.select(
                    col("storeId"),
                    col("name"),
                    col("salesPersonId"),
                    col("demographics.AnnualSales").alias("AnnualSales"),
                    col("demographics.AnnualRevenue").alias("AnnualRevenue"),
                    col("demographics.BankName").alias("BankName"),
                    col("demographics.BusinessType").alias("BusinessType"),
                    col("demographics.YearOpened").alias("YearOpened"),
                    col("demographics.Specialty").alias("Specialty"),
                    col("demographics.SquareFeet").alias("SquareFeet"),
                    col("demographics.Brands").alias("Brands"),
                    col("demographics.Internet").alias("Internet"),
                    col("demographics.NumberEmployees").alias("NumberEmployees"),
                    col("modifiedDate")
                )          
            # Write df_spark to Delta table in Databricks
            df.write\
                .format("delta")\
                .mode("overwrite")\
                .option("mergeSchema", "true")\
                .saveAsTable(tbl_full_name)
            print(f"DataFrame written to {tbl_full_name}")


        except Exception as e:
            print(f"Error writing to {tbl_full_name} - ", str(e.args))
            traceback.print_exc(e)

                


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp
import traceback

def main():

    try:
        df_converted = convert_json_to_dataframe()
        print(f"Complete Ingestion in Bronze Layer!")
    except Exception as e: # list possible exceptions (specific)
        print(f"Error in main - {str({e.args})}")
        traceback.print_exc(e)




In [0]:
# import pandas as pd
# def convert_json_to_dataframe(json_datas) -> pd.DataFrame:

#     # Flatten the JSON and convert to DataFrame
#     # json_dts = json.loads(json_datas)
#     flattened_data = pd.json_normalize(json_datas)
#     df_pd = pd.DataFrame(flattened_data)
#     return df_pd

    

In [0]:
# from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
# from typing import List
# def retrieve_schemas()-> List[StructType]:
#     adventureworks_schema = {
#         "addresses": StructType([
#             StructField("addressId", IntegerType(), False),       # Primary key, not nullable
#             StructField("addressLine1", StringType(), True),      # Nullable
#             StructField("addressLine2", StringType(), True),      # Nullable
#             StructField("city", StringType(), True),              # Nullable
#             StructField("state", StringType(), True),             # Nullable
#             StructField("country", StringType(), True),           # Nullable
#             StructField("postalCode", StringType(), True),        # Nullable
#             StructField("modifiedDate", StringType(), True)       # Nullable, assuming date-time as string
#         ]),
#         "customers": StructType([
#             StructField("customerId", IntegerType(), False),      # Primary key, not nullable
#             StructField("personId", IntegerType(), True),         # Nullable, foreign key
#             StructField("storeId", IntegerType(), True),          # Nullable, foreign key
#             StructField("territory", StringType(), True),         # Nullable, foreign key
#             StructField("accountNumber", StringType(), True),     # Nullable
#             StructField("modifiedDate", StringType(), True)       # Nullable, assuming date-time as string
#         ]),
#         "persons": StructType([
#             StructField("personId", IntegerType(), False),          # Primary key, not nullable
#             StructField("personType", StringType(), True),          # Nullable
#             StructField("nameStyle", StringType(), True),           # Nullable
#             StructField("title", StringType(), True),               # Nullable
#             StructField("firstName", StringType(), True),           # Nullable
#             StructField("middleName", StringType(), True),          # Nullable
#             StructField("lastName", StringType(), True),            # Nullable
#             StructField("suffix", StringType(), True),              # Nullable
#             StructField("emailPromotion", StringType(), True),      # Nullable
#             StructField("additionalContactInfo", StringType(), True), # Nullable
#             StructField("demographics", StructType([
#                 StructField("TotalPurchaseYTD", StringType(), True)
#             ]), True),
#             StructField("modifiedDate", StringType(), True)         # Nullable, assuming date-time as string
#         ]),
#         "products": StructType([
#             StructField("productId", IntegerType(), False),             # Primary key, not nullable
#             StructField("name", StringType(), True),                    # Nullable
#             StructField("productNumber", StringType(), True),           # Nullable
#             StructField("isManufactured", BooleanType(), True),         # Nullable
#             StructField("isSaleable", BooleanType(), True),             # Nullable
#             StructField("color", StringType(), True),                   # Nullable
#             StructField("safetyStockLevel", IntegerType(), True),       # Nullable
#             StructField("reorderPoint", IntegerType(), True),           # Nullable
#             StructField("standardCost", DoubleType(), True),            # Nullable
#             StructField("listPrice", DoubleType(), True),               # Nullable
#             StructField("size", StringType(), True),                    # Nullable
#             StructField("sizeUnit", StringType(), True),                # Nullable
#             StructField("weightUnit", StringType(), True),              # Nullable
#             StructField("weight", DoubleType(), True),                  # Nullable
#             StructField("daysToManufacture", IntegerType(), True),      # Nullable
#             StructField("productLine", StringType(), True),             # Nullable
#             StructField("class", StringType(), True),                   # Nullable
#             StructField("style", StringType(), True),                   # Nullable
#             StructField("subcategory", StringType(), True),             # Nullable
#             StructField("category", StringType(), True),                # Nullable
#             StructField("model", StringType(), True),                   # Nullable
#             StructField("sellStartDate", StringType(), True),           # Nullable, assuming date-time as string
#             StructField("sellEndDate", StringType(), True),             # Nullable, assuming date-time as string
#             StructField("discontinuedDate", StringType(), True),        # Nullable, assuming date-time as string
#             StructField("modifiedDate", StringType(), True)             # Nullable, assuming date-time as string
#         ]),
#         "productCategories": StructType([
#             StructField("productCategoryId", IntegerType(), False),     # Primary key, not nullable
#             StructField("name", StringType(), True),                    # Nullable
#             StructField("modifiedDate", StringType(), True)             # Nullable, assuming date-time as string
#         ]),
#         "productDescriptions": StructType([
#             StructField("productDescriptionId", IntegerType(), False),   # Primary key, not nullable
#             StructField("description", StringType(), True),              # Nullable
#             StructField("cultureCode", StringType(), True),              # Nullable
#             StructField("modifiedDate", StringType(), True)              # Nullable, assuming date-time as string
#         ]),
#         "productModels": StructType([
#             StructField("productModelId", IntegerType(), False),         # Primary key, not nullable
#             StructField("name", StringType(), True),                    # Nullable
#             StructField("catalogDescription", StringType(), True),      # Nullable
#             StructField("instructions", StringType(), True),            # Nullable
#             StructField("modifiedDate", StringType(), True)             # Nullable, assuming date-time as string
#         ]),
#         "salesOrderDetails": StructType([
#             StructField("salesOrderId", IntegerType(), False),          # Primary key, foreign key to SalesOrderHeader
#             StructField("salesOrderDetailId", IntegerType(), False),    # Primary key, not nullable
#             StructField("carrierTrackingNumber", StringType(), True),    # Nullable
#             StructField("orderQuantity", IntegerType(), True),          # Nullable
#             StructField("productId", IntegerType(), False),             # Foreign key to Product
#             StructField("productName", StringType(), True),             # Nullable
#             StructField("specialOfferId", IntegerType(), True),         # Nullable, foreign key to SpecialOffer
#             StructField("unitPrice", DoubleType(), True),               # Nullable
#             StructField("unitPriceDiscount", DoubleType(), True),       # Nullable
#             StructField("lineTotal", DoubleType(), True),               # Nullable
#             StructField("modifiedDate", StringType(), True)             # Nullable, assuming date-time as string
#         ]),
#         "salesOrders": StructType([
#             StructField("salesOrderId", IntegerType(), False),              # Primary key, not nullable
#             StructField("revisionNumber", IntegerType(), True),            # Nullable
#             StructField("orderDate", StringType(), True),                  # Nullable, assuming date-time as string
#             StructField("dueDate", StringType(), True),                    # Nullable, assuming date-time as string
#             StructField("shipDate", StringType(), True),                   # Nullable, assuming date-time as string
#             StructField("status", StringType(), True),                     # Nullable
#             StructField("isOrderedOnline", BooleanType(), True),           # Nullable
#             StructField("salesOrderNumber", StringType(), True),           # Nullable
#             StructField("purchaseOrderNumber", StringType(), True),        # Nullable
#             StructField("accountNumber", StringType(), True),              # Nullable
#             StructField("customerId", IntegerType(), False),               # Foreign key to Customer
#             StructField("salesPersonId", IntegerType(), True),             # Nullable, foreign key to SalesPerson
#             StructField("territory", StringType(), True),                  # Nullable, foreign key to SalesTerritory
#             StructField("billToAddress", StringType(), True),              # Nullable, foreign key to Address
#             StructField("shipToAddress", StringType(), True),              # Nullable, foreign key to Address
#             StructField("shipMethod", StringType(), True),                 # Nullable, foreign key to ShipMethod
#             StructField("creditCardId", IntegerType(), True),              # Nullable, foreign key to CreditCard
#             StructField("creditCardApprovalCode", StringType(), True),     # Nullable
#             StructField("currencyRateId", IntegerType(), True),            # Nullable, foreign key to CurrencyRate
#             StructField("subTotal", DoubleType(), True),                   # Nullable
#             StructField("taxAmount", DoubleType(), True),                  # Nullable
#             StructField("freight", DoubleType(), True),                    # Nullable
#             StructField("totalDue", DoubleType(), True),                   # Nullable
#             StructField("comment", StringType(), True),                    # Nullable
#             StructField("modifiedDate", StringType(), True)                # Nullable, assuming date-time as string
#         ]),
#         "stores": StructType([
#             StructField("storeId", IntegerType(), False),               # Primary key, foreign key to Customer
#             StructField("name", StringType(), True),                    # Nullable
#             StructField("salesPersonId", IntegerType(), True),          # Nullable, foreign key to SalesPerson
#             StructField("demographics", StructType([
#                 StructField("AnnualSales", StringType(), True),
#                 StructField("AnnualRevenue", StringType(), True),
#                 StructField("BankName", StringType(), True),
#                 StructField("BusinessType", StringType(), True),
#                 StructField("YearOpened", StringType(), True),
#                 StructField("Specialty", StringType(), True),
#                 StructField("SquareFeet", StringType(), True),
#                 StructField("Brands", StringType(), True),
#                 StructField("Internet", StringType(), True),
#                 StructField("NumberEmployees", StringType(), True),
#             ]), True),           # Nullable, assuming demographics stored as string
#             StructField("modifiedDate", StringType(), True)             # Nullable, assuming date-time as string
#         ])
#     }

#     return adventureworks_schema


In [0]:
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import col, current_timestamp

# def main():

#     spark = SparkSession.builder.appName("Ingest to Bronze Layer")\
#         .getOrCreate()

#     # Returns list of tables based on endpoint
#     tbl_list = [
#         "addresses", "customers", "products", "productCategories", "productDescriptions",
#         "productModels", "salesOrderDetails", "salesOrders", "stores", "persons"
#     ]

#     api_schemas_list = retrieve_schemas()
#     for tbl in tbl_list:
#         print(f"Processing {tbl} endpoint...")
#         json_dt = retrieve_api_data(tbl)
#         # print(json_dt)

#         # Call function to convert json to dataframe
#         df = convert_json_to_dataframe(json_dt)
#         # print(df)

#         if df.count == 0:
#             print(f"Empty DataFrame")
#         else:
#             # print(df)

#             schema = "bronze_adworks_jacq"  # Bronze schema name
#             catalog = "adventureworks_dev"  # Unity Catalog name
#             tbl_full_name = f"{catalog}.{schema}.{tbl}"
#             try:
                
#                 # Retrieve list of Schemas with StructTypes
#                 tbl_schema = api_schemas_list[tbl]

#                 # Create df_spark dataframe for Spark
#                 df_spark = spark.createDataFrame(df, tbl_schema)
#                 # For handling of Demographics in persons and stores tables
#                 if(tbl == "persons"):
#                     df_spark = df_spark.select(
#                         col("personId"),
#                         col("personType"),
#                         col("nameStyle"),
#                         col("title"),
#                         col("firstName"),
#                         col("middleName"),
#                         col("lastName"),
#                         col("suffix"),
#                         col("emailPromotion"),
#                         col("additionalContactInfo"),
#                         col("demographics.TotalPurchaseYTD").alias("TotalPurchaseYTD"),
#                         col("modifiedDate")
#                     )
#                 elif (tbl == "stores"):
#                     df_spark = df_spark.select(
#                         col("storeId"),
#                         col("name"),
#                         col("salesPersonId"),
#                         col("demographics.AnnualSales").alias("AnnualSales"),
#                         col("demographics.AnnualRevenue").alias("AnnualRevenue"),
#                         col("demographics.BankName").alias("BankName"),
#                         col("demographics.BusinessType").alias("BusinessType"),
#                         col("demographics.YearOpened").alias("YearOpened"),
#                         col("demographics.Specialty").alias("Specialty"),
#                         col("demographics.SquareFeet").alias("SquareFeet"),
#                         col("demographics.Brands").alias("Brands"),
#                         col("demographics.Internet").alias("Internet"),
#                         col("demographics.NumberEmployees").alias("NumberEmployees"),
#                         col("modifiedDate")
#                     )
                    
#                 # df_spark = df_spark.withColumn("updateDate", current_timestamp()) # Get the ingestion date
#                 df_spark.printSchema()

#                 # Write df_spark to Delta table in Databricks
#                 df_spark.write\
#                 .format("delta")\
#                 .mode("overwrite")\
#                 .option("mergeSchema", "true")\
#                 .saveAsTable(tbl_full_name)

#                 print(f"DataFrame written to {tbl_full_name}")
#             except Exception as e: # list possible exceptions (specific)
#                 print(f"Error writing DataFrame to {tbl_full_name} - {e}")





In [0]:
if __name__ == "__main__":
    main()

Processing addresses endpoint...
Successfully retrieve json_data from https://demodata.grapecity.com/adventureworks/api/v1/addresses
Converting addresses json data to DataFrame...
root
 |-- addressId: long (nullable = true)
 |-- addressLine1: string (nullable = true)
 |-- addressLine2: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- modifiedDate: string (nullable = true)
 |-- postalCode: string (nullable = true)
 |-- state: string (nullable = true)

addresses json data has been converted to DataFrame...
DataFrame written to adventureworks_dev.bronze_adworks_jaq_test.addresses
Processing customers endpoint...
Successfully retrieve json_data from https://demodata.grapecity.com/adventureworks/api/v1/customers
Converting customers json data to DataFrame...
root
 |-- accountNumber: string (nullable = true)
 |-- customerId: long (nullable = true)
 |-- modifiedDate: string (nullable = true)
 |-- personId: string (nullable = true)
 |-- s